Question 6: Identify features and target variables for both tasks

In [1]:
import pandas as pd

df = pd.read_csv("data.csv")

df_clean = df.drop(columns=['student_id', 'course_start_date'])

y_class = df_clean['completion_status']
y_reg = df_clean['final_score']         

X = df_clean.drop(columns=['completion_status', 'final_score'])

print(f"Features (X) shape: {X.shape}")

Features (X) shape: (5200, 15)


Question 7: Perform a train–test split for classification and regression problems

In [2]:
from sklearn.model_selection import train_test_split

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, stratify=y_class, random_state=42
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print(f"Classification Train size: {X_train_c.shape[0]}, Test size: {X_test_c.shape[0]}")
print(f"Regression Train size: {X_train_r.shape[0]}, Test size: {X_test_r.shape[0]}")

Classification Train size: 4160, Test size: 1040
Regression Train size: 4160, Test size: 1040


Question 8: Apply basic preprocessing (scaling/encoding if required)

In [3]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_train_c_processed = preprocessor.fit_transform(X_train_c)
X_test_c_processed = preprocessor.transform(X_test_c)

X_train_r_processed = preprocessor.fit_transform(X_train_r)
X_test_r_processed = preprocessor.transform(X_test_r)

print(f"Original Feature Count: {X.shape[1]}")
print(f"New Feature Count (After One-Hot Encoding): {X_train_c_processed.shape[1]}")


Original Feature Count: 15
New Feature Count (After One-Hot Encoding): 30


C:\Users\DELL\AppData\Local\Temp\ipykernel_28924\1713628988.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


# Part C: Bagging (Bootstrap Aggregating)

In [4]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train_c_processed, y_train_c)
acc_dt_c = accuracy_score(y_test_c, dt_clf.predict(X_test_c_processed))

bag_clf = BaggingClassifier(estimator=dt_clf, n_estimators=50, random_state=42)
bag_clf.fit(X_train_c_processed, y_train_c)
acc_bag_c = accuracy_score(y_test_c, bag_clf.predict(X_test_c_processed))

Question 10 & 11: Bagging Regressor vs Single Base Model

In [5]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train_r_processed, y_train_r)
r2_dt_r = r2_score(y_test_r, dt_reg.predict(X_test_r_processed))

bag_reg = BaggingRegressor(estimator=dt_reg, n_estimators=50, random_state=42)
bag_reg.fit(X_train_r_processed, y_train_r)
r2_bag_r = r2_score(y_test_r, bag_reg.predict(X_test_r_processed))

print("--- Classification (Completion Status) ---")
print(f"Single Tree Accuracy: {acc_dt_c:.4f}")
print(f"Bagging Accuracy:     {acc_bag_c:.4f}")

print("\n--- Regression (Final Score) ---")
print(f"Single Tree R²:       {r2_dt_r:.4f}")
print(f"Bagging R²:           {r2_bag_r:.4f}")

--- Classification (Completion Status) ---
Single Tree Accuracy: 0.6067
Bagging Accuracy:     0.7144

--- Regression (Final Score) ---
Single Tree R²:       -0.1234
Bagging R²:           0.4558


In [6]:
!pip install lightgbm xgboost

Defaulting to user installation because normal site-packages is not writeable
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   - -------------------------------------- 1.8/69.5 MB 14.3 MB/s eta 0:00:05
   --- ------------------------------------ 6.8/69.5 MB 20.0 MB/s eta 0:00:04
   ------- -------------------------------- 12.3/69.5 MB 22.0 MB/s eta 0:00:03
   --------- ------------------------------ 16.0/69.5 MB 20.5 MB/s eta 0:00:03
   ------------ --------------------------- 21.8/69.5 MB 22.6 MB/s eta 0:00:03
   --------------- ------------------------ 27.0/69.5 MB 22.8 MB/s eta 0:00:02
   ---------------- ----------------------- 28.3/69.5 MB 19.7 MB/s eta 0:00:03
   ---------------- ----------------------- 28.8/69.5 MB 19.2 MB/s eta 0:00:03
   ----------------- ---------------------- 30.9/69.5 MB 16.6 MB/s eta 0:00:03
   ----------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Part D: Boosting Algorithms

Questions 12 & 13: AdaBoost Classifier & Regressor



In [9]:
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.impute import SimpleImputer

# AdaBoost Classifier
ada_clf = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42)
imp = SimpleImputer(strategy='mean')
X_train_c_processed = imp.fit_transform(X_train_c_processed)
X_test_c_processed = imp.transform(X_test_c_processed)

# also impute regression processed arrays to avoid NaNs later
imp_r = SimpleImputer(strategy='mean')
X_train_r_processed = imp_r.fit_transform(X_train_r_processed)
X_test_r_processed = imp_r.transform(X_test_r_processed)

ada_clf.fit(X_train_c_processed, y_train_c)
acc_ada_c = accuracy_score(y_test_c, ada_clf.predict(X_test_c_processed))

# AdaBoost Regressor
ada_reg = AdaBoostRegressor(n_estimators=50, learning_rate=1.0, random_state=42)
ada_reg.fit(X_train_r_processed, y_train_r)
r2_ada_r = r2_score(y_test_r, ada_reg.predict(X_test_r_processed))

print(f"AdaBoost Accuracy: {acc_ada_c:.4f} | AdaBoost R²: {r2_ada_r:.4f}")

AdaBoost Accuracy: 0.7260 | AdaBoost R²: 0.4029


Questions 15 & 16: Gradient Boosting Classifier & Regressor

In [13]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.impute import SimpleImputer

# Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
imp_gb = SimpleImputer(strategy='mean')

X_train_c_gb = imp_gb.fit_transform(X_train_c_processed)
X_test_c_gb = imp_gb.transform(X_test_c_processed)

gb_clf.fit(X_train_c_gb, y_train_c)
acc_gb_c = accuracy_score(y_test_c, gb_clf.predict(X_test_c_gb))

imp_gb_r = SimpleImputer(strategy='mean')

X_train_r_gb = imp_gb_r.fit_transform(X_train_r_processed)
X_test_r_gb = imp_gb_r.transform(X_test_r_processed)

# Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
imp_gb = SimpleImputer(strategy='mean')

X_train_c_gb = imp_gb.fit_transform(X_train_c_processed)
X_test_c_gb = imp_gb.transform(X_test_c_processed)

gb_clf.fit(X_train_c_gb, y_train_c)
acc_gb_c = accuracy_score(y_test_c, gb_clf.predict(X_test_c_gb))

# Gradient Boosting Regressor
imp_gb_r = SimpleImputer(strategy='mean')

X_train_r_gb = imp_gb_r.fit_transform(X_train_r_processed)
X_test_r_gb = imp_gb_r.transform(X_test_r_processed)

gb_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_reg.fit(X_train_r_gb, y_train_r)
r2_gb_r = r2_score(y_test_r, gb_reg.predict(X_test_r_gb))

print(f"GB Accuracy: {acc_gb_c:.4f} | GB R²: {r2_gb_r:.4f}")
r2_gb_r = r2_score(y_test_r, gb_reg.predict(X_test_r_gb))

print(f"GB Accuracy: {acc_gb_c:.4f} | GB R²: {r2_gb_r:.4f}")
acc_gb_c = accuracy_score(y_test_c, gb_clf.predict(X_test_c_processed))

# Gradient Boosting Regressor
gb_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_reg.fit(X_train_r_processed, y_train_r)
r2_gb_r = r2_score(y_test_r, gb_reg.predict(X_test_r_processed))

print(f"GB Accuracy: {acc_gb_c:.4f} | GB R²: {r2_gb_r:.4f}")

GB Accuracy: 0.7346 | GB R²: 0.4802
GB Accuracy: 0.7346 | GB R²: 0.4802
GB Accuracy: 0.7346 | GB R²: 0.4802


Questions 18 & 19: LightGBM Classifier & Regressor

In [10]:
from lightgbm import LGBMClassifier, LGBMRegressor

lgb_clf = LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
lgb_clf.fit(X_train_c_processed, y_train_c)

lgb_reg = LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
lgb_reg.fit(X_train_r_processed, y_train_r)

print(f"LGBM Accuracy: {accuracy_score(y_test_c, lgb_clf.predict(X_test_c_processed)):.4f}")
print(f"LGBM R²:       {r2_score(y_test_r, lgb_reg.predict(X_test_r_processed)):.4f}")


LGBM Accuracy: 0.7250
LGBM R²:       0.4590


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Questions 21 & 22: XGBoost Classifier & Regressor

In [12]:
from xgboost import XGBClassifier, XGBRegressor

xgb_clf = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_clf.fit(X_train_c_processed, y_train_c)

xgb_reg = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_reg.fit(X_train_r_processed, y_train_r)

print(f"XGBoost Accuracy: {accuracy_score(y_test_c, xgb_clf.predict(X_test_c_processed)):.4f}")
print(f"XGBoost R²:       {r2_score(y_test_r, xgb_reg.predict(X_test_r_processed)):.4f}")


XGBoost Accuracy: 0.7202
XGBoost R²:       0.4669


# Part E: Voting & Stacking Ensembles

Question 24 & 25: Voting Classifier (Hard vs Soft)

In [15]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Base estimators
clf1 = LogisticRegression(random_state=42)
clf2 = SVC(probability=True, random_state=42)
clf3 = KNeighborsClassifier(n_neighbors=5)

# Hard Voting: Majority rule
hard_voting = VotingClassifier(estimators=[('lr', clf1), ('svc', clf2), ('knn', clf3)], voting='hard')
hard_voting.fit(X_train_c_processed, y_train_c)

# Soft Voting: Average of probabilities
soft_voting = VotingClassifier(estimators=[('lr', clf1), ('svc', clf2), ('knn', clf3)], voting='soft')
soft_voting.fit(X_train_c_processed, y_train_c)

print(f"Hard Voting Accuracy: {accuracy_score(y_test_c, hard_voting.predict(X_test_c_processed)):.4f}")
print(f"Soft Voting Accuracy: {accuracy_score(y_test_c, soft_voting.predict(X_test_c_processed)):.4f}")


Hard Voting Accuracy: 0.7288
Soft Voting Accuracy: 0.7327


Question 26: Stacking Classifier

In [16]:
from sklearn.ensemble import StackingClassifier

# Define base models
base_learners = [('lr', clf1), ('knn', clf3)]

# Define meta-learner
stack_clf = StackingClassifier(estimators=base_learners, final_estimator=LogisticRegression())
stack_clf.fit(X_train_c_processed, y_train_c)

print(f"Stacking Classifier Accuracy: {accuracy_score(y_test_c, stack_clf.predict(X_test_c_processed)):.4f}")


Stacking Classifier Accuracy: 0.7462


Question 27: Stacking Regressor

In [17]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Define base models
base_regressors = [('rf', RandomForestRegressor(n_estimators=50)), ('ridge', Ridge())]

# Define meta-learner
stack_reg = StackingRegressor(estimators=base_regressors, final_estimator=Ridge())
stack_reg.fit(X_train_r_processed, y_train_r)

print(f"Stacking Regressor R²: {r2_score(y_test_r, stack_reg.predict(X_test_r_processed)):.4f}")


Stacking Regressor R²: 0.4937


 #  Part F: Model Evaluation & Comparison

Question 28 & 29: Model Evaluation Metrics

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Classifier list (Assuming they were trained in Part D/E)
classifiers = [('AdaBoost', ada_clf), ('GBoost', gb_clf), ('LGBM', lgb_clf), ('XGB', xgb_clf), ('Stacking', stack_clf)]
regressors = [('AdaBoost', ada_reg), ('GBoost', gb_reg), ('LGBM', lgb_reg), ('XGB', xgb_reg), ('Stacking', stack_reg)]

# Evaluation loop for Classifiers
class_results = []
for name, model in classifiers:
    preds = model.predict(X_test_c_processed)
    probs = model.predict_proba(X_test_c_processed)[:, 1]
    class_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, preds),
        "Precision": precision_score(y_test_c, preds),
        "Recall": recall_score(y_test_c, preds),
        "F1-Score": f1_score(y_test_c, preds),
        "ROC-AUC": roc_auc_score(y_test_c, probs)
    })

# Evaluation loop for Regressors
reg_results = []
for name, model in regressors:
    preds = model.predict(X_test_r_processed)
    reg_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_r, preds),
        "RMSE": mean_squared_error(y_test_r, preds) ** 0.5,
        "R2 Score": r2_score(y_test_r, preds)
    })

print("--- Classification Metrics ---")
print(pd.DataFrame(class_results).to_string(index=False))
print("\n--- Regression Metrics ---")
print(pd.DataFrame(reg_results).to_string(index=False))

C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


--- Classification Metrics ---
   Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC
AdaBoost  0.725962   0.661538 0.551282  0.601399 0.784761
  GBoost  0.734615   0.671687 0.571795  0.617729 0.790631
    LGBM  0.725000   0.648571 0.582051  0.613514 0.768840
     XGB  0.720192   0.641026 0.576923  0.607287 0.774706
Stacking  0.746154   0.692073 0.582051  0.632312 0.793763

--- Regression Metrics ---
   Model      MAE      RMSE  R2 Score
AdaBoost 8.571876 10.562639  0.402860
  GBoost 7.907590  9.854802  0.480211
    LGBM 8.118179 10.054104  0.458974
     XGB 7.987363  9.980059  0.466914
Stacking 7.816128  9.726218  0.493687


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
